The goal of this notebook is to investigate and quantify the change in each SDP pigment over the lifetimes of cyclones traveling South and anticyclones traveling North respectively. The pigments are the 13 concentrations that the Kramer et al. (2022) Spectral Derivative Pigments model retrieves from the 8-day PACE Rrs composite inside each eddy: total chlorophyll-a and 12 accessory pigments, in mg/m³ (SDP reports µg/L, which is the same number). SDP clips a negative prediction to zero, and those pixels stay in the means.

Target eddies are the same as in `chl_over_lifetime.ipynb`, and the coverage rule is the one of the SDP stages:
- Cyclones formed north of the Gulf Stream axis and ended south, or formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south. Anticyclones use the reversed rule.
- 80% Rrs coverage of the speed-contour interior and at least 10 valid pixels per eddy-composite, from `collocate_pace` and `build_gold_table`. PACE begins in March 2024, so the record is shorter than the Copernicus one and fewer target eddies have a composite.

Two views:
- Over lifetime: the interior mean of each pigment per age bin, one value per eddy, like the third figure of `chl_over_lifetime.ipynb`.
- Age and radius: the mean of each pigment in each age bin and each 0.2 R ring from the eddy center out to 2 radii. The rings are circles around the track center that `collocate_pace` matched to the composite, R is the PET speed radius, and the pixels are the SDP retrievals that the stage collected out to `max_radius` speed radii of the center or inside the speed contour. A ring needs 3 valid pixels, and a cell needs 3 eddies.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_AGE_BINS = 5
N_RADIAL_BINS = 10
MAX_RADIUS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
EXCLUDE_RECORD_EDGE_TRACKS = False
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']
pigments = ['Tchla', 'Zea', 'DV_chla', 'ButFuco', 'HexFuco', 'Allo', 'MV_chlb', 'Neo', 'Viola', 'Fuco', 'Chlc12', 'Chlc3', 'Perid']
pigment_labels = {
    'Tchla': 'Total chlorophyll-a', 'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
    'ButFuco': "19'-But-fucoxanthin", 'HexFuco': "19'-Hex-fucoxanthin", 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin', 'Fuco': 'Fucoxanthin',
    'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Perid': 'Peridinin',
}
panel_letters = 'abcdefghijklm'

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigment_table = pigment_table.rename(columns={f'eddy_mean_{pigment}': pigment for pigment in pigments})
physical_start, physical_end = pd.to_datetime(cfg['base']['time']['eddy_date_range'])
eddy_tracks['at_record_edge'] = (
    (eddy_tracks['birth_date'] <= physical_start)
    | (eddy_tracks['death_date'] >= physical_end)
)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
pigment_table = pigment_table.merge(
    eddy_tracks[identity_columns + ['at_record_edge', 'is_target']],
    on=identity_columns, how='left',
)
target_composites = cast(pd.DataFrame, pigment_table.loc[pigment_table['is_target']]).copy()
if EXCLUDE_RECORD_EDGE_TRACKS:
    target_composites = cast(pd.DataFrame, target_composites.loc[~target_composites['at_record_edge']]).copy()

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

print(f'Target eddies and composites in the SDP pigment table: the distinct eddies of each polarity and the eddy-composites that pass the Rrs rule of {cfg["collocate_pace"]["min_coverage"]:.0%} coverage of the speed-contour interior and at least 10 valid pixels. Composite midpoints run from {target_composites["date"].min():%Y-%m-%d} to {target_composites["date"].max():%Y-%m-%d}, inside the PACE record. {len(target_composites.drop_duplicates(identity_columns))} of the {int(eddy_tracks["is_target"].sum())} target eddies have at least one composite.')
display(cast(pd.DataFrame, target_composites.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
)))

In [ ]:
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
analysis = target_composites.sort_values(identity_columns + ['date']).melt(
    id_vars=identity_columns + ['date', 'age_frac'], value_vars=pigments,
    var_name='pigment', value_name='concentration',
)
analysis['age_bin'] = np.minimum(
    (analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
)
analysis['change'] = analysis['concentration'] - analysis.groupby(identity_columns + ['pigment'])['concentration'].transform('first')
eddy_bins = cast(pd.DataFrame, analysis.groupby(identity_columns + ['pigment', 'age_bin']).agg(
    concentration=('concentration', 'mean'), change=('change', 'mean'),
)).reset_index()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for polarity in polarity_names:
    polarity_bins = eddy_bins.loc[eddy_bins['polarity'].eq(polarity)]
    eddy_ids = sorted(polarity_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for pigment in pigments:
        pigment_bins = polarity_bins.loc[polarity_bins['pigment'].eq(pigment)]
        for metric in ('concentration', 'change'):
            matrix = pigment_bins.pivot(index='track_id', columns='age_bin', values=metric).reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
            counts = np.isfinite(matrix).sum(axis=0)
            means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
            sampled = matrix[draws]
            sampled_counts = np.isfinite(sampled).sum(axis=1)
            sampled_means = np.divide(
                np.nansum(sampled, axis=1), sampled_counts,
                out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0,
            )
            for age_bin in range(N_AGE_BINS):
                bootstrap_values = sampled_means[:, age_bin]
                bootstrap_values = bootstrap_values[np.isfinite(bootstrap_values)]
                low, high = (np.quantile(bootstrap_values, [0.025, 0.975]) if counts[age_bin] >= 3 else (np.nan, np.nan))
                summary_rows.append({
                    'polarity': polarity, 'pigment': pigment, 'metric': metric, 'age_bin': age_bin,
                    'age_midpoint': bin_centers[age_bin], 'mean': means[age_bin],
                    'ci_low': low, 'ci_high': high, 'n_eddies': int(counts[age_bin]),
                })
lifetime_summary = pd.DataFrame(summary_rows)
n_eddies = analysis.groupby('polarity')['track_id'].nunique()
bin_counts = lifetime_summary.loc[lifetime_summary['metric'].eq('concentration') & lifetime_summary['pigment'].eq('Tchla')].pivot(index='polarity', columns='age_bin', values='n_eddies')

for metric, ylabel in (
    ('concentration', 'Pigment concentration (mg m$^{-3}$)'),
    ('change', 'Change from first observation (mg m$^{-3}$)'),
):
    life_fig, life_axes = cast(tuple[Figure, np.ndarray], plt.subplots(4, 4, figsize=(6.69, 7.6), sharex=True, layout='constrained'))
    for ax in life_axes.flat[len(pigments) + 1:]:
        cast(Axes, ax).remove()
    for letter, ax, pigment in zip(panel_letters, life_axes.flat, pigments):
        ax = cast(Axes, ax)
        if metric == 'change':
            ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
        for polarity in polarity_names:
            result = lifetime_summary.loc[
                lifetime_summary['polarity'].eq(polarity) & lifetime_summary['pigment'].eq(pigment) & lifetime_summary['metric'].eq(metric)
            ].sort_values('age_bin')
            color = polarity_colors[polarity]
            intervals = result.loc[result['ci_low'].notna()]
            ax.errorbar(
                intervals['age_midpoint'], intervals['mean'],
                yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
                fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=2,
            )
            ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3, markeredgecolor='white', markeredgewidth=0.5, label=f'{target_labels[polarity]} (n = {n_eddies[polarity]})', zorder=3)
        ax.set_title(f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', loc='left', fontsize=8)
        ax.xaxis.set_ticks(np.linspace(0, 1, 6))
        ax.set_xlim(0, 1)
        ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
        locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
        ax.yaxis.set_major_locator(locator)
        ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
        ax.set_ylim(ticks[0], ticks[-1])
    for ax in (life_axes[-1, 0], *life_axes[-2, 1:]):
        ax = cast(Axes, ax)
        ax.tick_params(labelbottom=True)
    legend_ax = cast(Axes, life_axes.flat[len(pigments)])
    legend_ax.axis('off')
    legend_ax.legend(*cast(Axes, life_axes.flat[0]).get_legend_handles_labels(), loc='upper left', fontsize=8).set_in_layout(False)
    life_fig.supxlabel('Fraction of observed track', fontsize=8)
    life_fig.supylabel(ylabel, fontsize=8)
    plt.show()
print('Mean pigment concentration per bin, the values of the first figure, in mg/m³.')
display(lifetime_summary.loc[lifetime_summary['metric'].eq('concentration')].pivot(index='pigment', columns=['polarity', 'age_bin'], values='mean').reindex(pigments).round(4))
print('Eddies per bin. The 13 pigments come from the same pixels, so the count is the same for each.')
display(bin_counts)

In [ ]:
radial_edges = np.linspace(0, MAX_RADIUS, N_RADIAL_BINS + 1)
pixel_columns = {'T chla': 'Tchla', 'DV chla': 'DV_chla', 'MV chlb': 'MV_chlb', 'chl c1+c2': 'Chlc12', 'chl c3': 'Chlc3'}
pixels = pd.concat([
    pd.read_parquet(DATA_DIR / f'silver/pigments/{polarity}/eddy_{track_id}_pigments.parquet').assign(polarity=polarity)
    for polarity, track_id in target_composites[identity_columns].drop_duplicates().itertuples(index=False)
], ignore_index=True).rename(columns=pixel_columns)
pixels = pixels.merge(target_composites[identity_columns + ['date', 'age_frac']], on=identity_columns + ['date'])
half_chord = (
    np.sin(np.radians(pixels['pixel_lat'] - pixels['center_lat']) / 2) ** 2
    + np.cos(np.radians(pixels['pixel_lat'])) * np.cos(np.radians(pixels['center_lat'])) * np.sin(np.radians(pixels['pixel_lon'] - pixels['center_lon']) / 2) ** 2
)
pixels['radial_bin'] = np.digitize(2 * 6371 * np.arcsin(np.sqrt(half_chord)) / pixels['radius_km'], radial_edges) - 1
rings = cast(pd.DataFrame, pixels.loc[pixels['radial_bin'].lt(N_RADIAL_BINS)].groupby(identity_columns + ['date', 'age_frac', 'radial_bin']).agg(
    n_pixels=('Tchla', 'size'), **{pigment: (pigment, 'mean') for pigment in pigments},
)).reset_index()
rings.loc[rings['n_pixels'].lt(3), pigments] = np.nan
radial_analysis = rings.melt(
    id_vars=identity_columns + ['date', 'age_frac', 'radial_bin'], value_vars=pigments,
    var_name='pigment', value_name='concentration',
)
radial_analysis['age_bin'] = np.minimum(
    (radial_analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
)
eddy_cells = radial_analysis.groupby(identity_columns + ['pigment', 'age_bin', 'radial_bin'])['concentration'].mean().reset_index()
cells = cast(pd.DataFrame, eddy_cells.groupby(['polarity', 'pigment', 'age_bin', 'radial_bin']).agg(
    concentration=('concentration', 'mean'), n_eddies=('concentration', 'count'),
)).reset_index()
cells.loc[cells['n_eddies'].lt(3), 'concentration'] = np.nan
cell_counts = cells.loc[cells['pigment'].eq('Tchla')].pivot(index='radial_bin', columns=['polarity', 'age_bin'], values='n_eddies')

cmap = plt.get_cmap('viridis').copy()
cmap.set_bad('#e6e6e6')
radial_fig = plt.figure(figsize=(6.69, 8.9))
outer = radial_fig.add_gridspec(5, 3, left=0.075, right=0.92, bottom=0.045, top=0.96, wspace=0.6, hspace=0.6)
for index, (letter, pigment) in enumerate(zip(panel_letters, pigments)):
    pigment_cells = cells.loc[cells['pigment'].eq(pigment)]
    vmin, vmax = pigment_cells['concentration'].min(), pigment_cells['concentration'].max()
    axes = cast(np.ndarray, outer[index].subgridspec(1, 2, wspace=0.12).subplots(sharey=True))
    for ax, polarity in zip(axes, polarity_names):
        ax = cast(Axes, ax)
        grid = pigment_cells.loc[pigment_cells['polarity'].eq(polarity)].pivot(index='radial_bin', columns='age_bin', values='concentration').reindex(index=range(N_RADIAL_BINS), columns=range(N_AGE_BINS)).to_numpy(dtype=float)
        ax.pcolormesh(bin_edges, radial_edges, np.ma.masked_invalid(grid), cmap=cmap, vmin=vmin, vmax=vmax, edgecolors='white', linewidth=0.5)
        ax.axhline(1, color='#222222', linewidth=0.8, linestyle=(0, (4, 2.5)), zorder=3)
        ax.set_aspect('equal')
        ax.set_title(f'{polarity.capitalize()}s', fontsize=8, pad=2)
        ax.xaxis.set_ticks([0, 0.5, 1], ['0', '0.5', '1'])
        ax.xaxis.set_ticks(bin_edges, minor=True)
        ax.yaxis.set_ticks([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1', '1.5', '2'])
        ax.yaxis.set_ticks(radial_edges, minor=True)
        ax.tick_params(labelsize=8, length=2)
        ax.tick_params(which='minor', length=1.2)
    colorbar = radial_fig.colorbar(ScalarMappable(norm=Normalize(vmin, vmax), cmap=cmap), cax=cast(Axes, axes[1]).inset_axes((1.12, 0, 0.1, 1)))
    colorbar.ax.tick_params(labelsize=8, length=2)
    colorbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
    colorbar.outline.set_linewidth(0.5)
    cast(Axes, axes[0]).text(0, 1.14, f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', transform=axes[0].transAxes, fontsize=8, va='bottom', ha='left')
radial_fig.supxlabel('Fraction of observed track', fontsize=8)
radial_fig.supylabel('Distance from eddy center (speed radii)', fontsize=8)
plt.show()
print('Eddies per cell by ring (rows) and age bin (columns). The 13 pigments come from the same pixels, so the count is the same for each.')
display(cell_counts)

In [ ]:
import io
from contextlib import redirect_stdout

import xarray as xr
from eddy_tracking.config import resolve_data_dir
from eddy_tracking.packages.sdp import PIGMENTS, run_sdp_on_pace_l3
from eddy_tracking.preprocess.ancillary import read_ancillary_grids
from eddy_tracking.preprocess.pace import parse_pace_window
from eddy_tracking.preprocess.streamline import KM_PER_DEG_LAT

DISTANCE_BIN_KM = 50
MAX_DISTANCE_KM = 3 * NEAR_AXIS_KM
PIXELS_PER_BIN = 40
distance_edges = np.arange(-MAX_DISTANCE_KM, MAX_DISTANCE_KM + 1, DISTANCE_BIN_KM)
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2
sides = {'south': (-MAX_DISTANCE_KM, -NEAR_AXIS_KM), 'north': (NEAR_AXIS_KM, MAX_DISTANCE_KM)}
side_colors = {'south': '#a6a6a6', 'north': '#444444'}
streamline = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet')
lon_origin = cfg['base']['region']['lon_range'][0]
streamline['lon_bin'] = ((streamline['lon'] - lon_origin) // 0.25).astype(int)
axis_lat = streamline.groupby(['date', 'lon_bin'])['lat'].mean().unstack('lon_bin')
sst_grid, sss_grid = read_ancillary_grids(resolve_data_dir(cfg, 'sst_dir'), resolve_data_dir(cfg, 'sss_dir'))
rng = np.random.default_rng(RANDOM_SEED)
profile_composites = []
for path in sorted(resolve_data_dir(cfg, 'pace_dir').glob('*.nc')):
    window = parse_pace_window(path.name, cfg['collocate_pace']['temporal_resolution'])
    if window is None or pd.Timestamp(window[0]) not in axis_lat.index:
        continue
    composite_date = pd.Timestamp(window[0])
    with xr.open_dataset(path) as ds:
        lon = ds['lon'].to_numpy()
        lat = ds['lat'].to_numpy()
        wavelengths = ds.coords['wavelength'].to_numpy().astype(int)
        rrs = ds['Rrs'].to_numpy().reshape(-1, ds.sizes['wavelength'])  # (n_lat, n_lon, n_wavelengths) -> (n_lat * n_lon, n_wavelengths)
    axis_at_lon = axis_lat.loc[composite_date].reindex(((lon - lon_origin) // 0.25).astype(int)).to_numpy()  # (n_lon,)
    offset_km = ((lat[:, None] - axis_at_lon[None, :]) * KM_PER_DEG_LAT).ravel()  # (n_lat, n_lon) -> (n_lat * n_lon,)
    distance_bin = np.digitize(offset_km, distance_edges) - 1
    candidate = np.isfinite(offset_km) & (distance_bin >= 0) & (distance_bin < len(distance_centers)) & np.all(np.isfinite(rrs), axis=1)
    picked = np.concatenate([
        rng.choice(members, size=min(PIXELS_PER_BIN, len(members)), replace=False)
        for members in (np.flatnonzero(candidate & (distance_bin == index)) for index in range(len(distance_centers)))
        if len(members)
    ])
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    observations = pd.concat([
        pd.DataFrame({'date': composite_date, 'pixel_lon': lon_grid.ravel()[picked], 'pixel_lat': lat_grid.ravel()[picked], 'distance_bin': distance_bin[picked]}),
        pd.DataFrame(rrs[picked], columns=[f'Rrs_{wavelength}' for wavelength in wavelengths]),  # pyright: ignore[reportArgumentType]
    ], axis=1)
    with redirect_stdout(io.StringIO()):
        retrieved, kept = run_sdp_on_pace_l3(observations, sst_grid, sss_grid)
    retrieved = retrieved.rename(columns=PIGMENTS)
    retrieved['distance_bin'] = kept['distance_bin'].to_numpy()
    profile_composites.append(retrieved.groupby('distance_bin')[pigments].mean().reindex(range(len(distance_centers))).to_numpy())
profile = np.stack(profile_composites).transpose(0, 2, 1)  # (n_composites, n_distance_bins, n_pigments) -> (n_composites, n_pigments, n_distance_bins)
profile_quantiles = np.nanpercentile(profile, [25, 50, 75], axis=0)  # (3, n_pigments, n_distance_bins)
side_bins = {side: (distance_centers > low) & (distance_centers < high) for side, (low, high) in sides.items()}
side_levels = pd.DataFrame(
    {(side, quantile): np.nanpercentile(np.nanmean(profile[:, :, bins], axis=2), quantile, axis=0) for side, bins in side_bins.items() for quantile in (25, 50, 75)},
    index=pd.Index(pigments),
)

profile_fig, profile_axes = cast(tuple[Figure, np.ndarray], plt.subplots(4, 4, figsize=(6.69, 7.6), sharex=True, layout='constrained'))
for ax in profile_axes.flat[len(pigments):]:
    cast(Axes, ax).remove()
for letter, ax, pigment_index, pigment in zip(panel_letters, profile_axes.flat, range(len(pigments)), pigments):
    ax = cast(Axes, ax)
    for side, (low, high) in sides.items():
        ax.axvspan(low, high, color=side_colors[side], alpha=0.12, linewidth=0, zorder=0)
    ax.axvline(0, color='#999999', linewidth=0.6, zorder=1)
    ax.fill_between(distance_centers, profile_quantiles[0, pigment_index], profile_quantiles[2, pigment_index], color='#444444', alpha=0.15, linewidth=0)
    ax.plot(distance_centers, profile_quantiles[1, pigment_index], color='#444444', linewidth=1.2)
    ax.set_title(f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', loc='left', fontsize=8)
    ax.set_xlim(-MAX_DISTANCE_KM, MAX_DISTANCE_KM)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.set_ylim(0, ticks[-1])
for ax in (profile_axes[-1, 0], *profile_axes[-2, 1:]):
    ax = cast(Axes, ax)
    ax.tick_params(labelbottom=True)
    ax.set_xlabel('Distance from axis (km)')
profile_fig.supylabel(f'Pigment concentration over {len(profile)} composites, {PIXELS_PER_BIN} pixels per bin (mg m$^{{-3}}$)', fontsize=8)
plt.show()

level_fig, level_axes = cast(tuple[Figure, np.ndarray], plt.subplots(4, 4, figsize=(6.69, 7.6), sharex=True, layout='constrained'))
for ax in level_axes.flat[len(pigments) + 1:]:
    cast(Axes, ax).remove()
for letter, ax, pigment in zip(panel_letters, level_axes.flat, pigments):
    ax = cast(Axes, ax)
    for side, (low, high) in sides.items():
        ax.axhspan(side_levels.loc[pigment, (side, 25)], side_levels.loc[pigment, (side, 75)], color=side_colors[side], alpha=0.18, linewidth=0, zorder=1)
        ax.axhline(side_levels.loc[pigment, (side, 50)], color=side_colors[side], linewidth=1.0, zorder=2, label=f'{side.capitalize()} of the axis, {min(abs(low), abs(high))} to {max(abs(low), abs(high))} km')
    for polarity in polarity_names:
        result = lifetime_summary.loc[
            lifetime_summary['polarity'].eq(polarity) & lifetime_summary['pigment'].eq(pigment) & lifetime_summary['metric'].eq('concentration')
        ].sort_values('age_bin')
        color = polarity_colors[polarity]
        intervals = result.loc[result['ci_low'].notna()]
        ax.errorbar(
            intervals['age_midpoint'], intervals['mean'],
            yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
            fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=3,
        )
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, linewidth=1.2, markersize=3, markeredgecolor='white', markeredgewidth=0.5, label=f'{target_labels[polarity]} (n = {n_eddies[polarity]})', zorder=4)
    ax.set_title(f'$\\bf{{({letter})}}$ {pigment_labels[pigment]}', loc='left', fontsize=8)
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    ax.set_ylim(0, ticks[-1])
for ax in (level_axes[-1, 0], *level_axes[-2, 1:]):
    ax = cast(Axes, ax)
    ax.tick_params(labelbottom=True)
legend_ax = cast(Axes, level_axes.flat[len(pigments)])
legend_ax.axis('off')
legend_ax.legend(*cast(Axes, level_axes.flat[0]).get_legend_handles_labels(), loc='upper left', fontsize=8).set_in_layout(False)
level_fig.supxlabel('Fraction of observed track', fontsize=8)
level_fig.supylabel('Pigment concentration (mg m$^{-3}$)', fontsize=8)
plt.show()
display(side_levels.round(4))